# 8-agent round-robin training — Live Loss

共有libcg＋CPUバッチ学習を別プロセスで起動し、バッチごとのLossをこのNotebook内で更新表示します。設定セルを確認してから、最後のセルを実行してください。

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
import threading
import time

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / 'tools' / 'run_train_round_robin.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'tools' / 'run_train_round_robin.py').exists():
    raise RuntimeError('pokemon-tcg-agentの配下からNotebookを起動してください。')

AGENTS = [f'rl_mcts_match_{index:02d}' for index in range(8)]
TARGET_LOSS = 0.03
RUN_DIR = ROOT / 'results' / 'train_round_robin_central' / f'notebook_{int(time.time())}'
REFRESH_SECONDS = 1.0

COMMAND = [
    sys.executable, '-u', str(ROOT / 'tools' / 'run_train_round_robin.py'),
    *[item for name in AGENTS for item in ('--agent', str(ROOT / 'agents' / name))],
    '--backend', 'shared-cpu-batch',
    '--iterations', '12', '--games', '5', '--search-count', '10',
    '--batch-size', '128', '--inference-batch-size', '128', '--lanes', '128',
    '--device', 'cpu', '--inference-device', 'cpu', '--eval-games', '0',
    '--target-loss', str(TARGET_LOSS), '--min-iterations', '3', '--loss-patience', '2',
    '--run-dir', str(RUN_DIR), '--live-loss',
]

In [ ]:
print('Run directory:', RUN_DIR)
print('Command:')
print(' '.join(COMMAND))
print('\nこのセルではまだ学習は開始していません。次のセルで開始します。')

In [ ]:
def read_loss_rows():
    path = RUN_DIR / 'live_loss_batches.csv'
    if not path.exists():
        return []
    try:
        with path.open(encoding='utf-8') as file:
            return list(csv.DictReader(file))
    except (OSError, csv.Error):
        return []


def read_status():
    path = RUN_DIR / 'live_status.json'
    if not path.exists():
        return {'phase': 'starting', 'message': '学習プロセスを起動中'}
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return {'phase': 'updating', 'message': '状態を更新中'}


def draw_live_loss(rows, status, log_lines, return_code=None):
    clear_output(wait=True)
    figure, axis = plt.subplots(figsize=(13, 7))
    latest = {}
    for name in AGENTS:
        agent_rows = [row for row in rows if row['agent'] == name]
        if not agent_rows:
            continue
        x_values = [int(row['iteration']) + int(row['batch']) / max(int(row['batches']), 1) for row in agent_rows]
        running = [float(row['running_loss']) for row in agent_rows]
        axis.plot(x_values, running, linewidth=2, label=name)
        latest[name] = agent_rows[-1]
    axis.axhline(TARGET_LOSS, color='red', linestyle='--', linewidth=2, label=f'target={TARGET_LOSS:g}')
    axis.set_title('Round-robin Live Loss (batch running average)')
    axis.set_xlabel('Iteration + batch progress')
    axis.set_ylabel('Loss')
    axis.set_ylim(bottom=0)
    axis.grid(alpha=0.25)
    axis.legend(ncol=3, fontsize=9)
    figure.tight_layout()
    display(figure)
    plt.close(figure)

    phase = status.get('phase', 'unknown')
    message = status.get('message', '')
    print(f'Status: {phase} — {message}')
    print(f"Iteration: {status.get('iteration', '—')}  Agent: {status.get('agent') or '—'}  Batch: {status.get('batch', 0)}/{status.get('batches', 0)}")
    print('')
    print(f"{'Agent':22s} {'Batch loss':>12s} {'Running avg':>12s} {'Value':>12s} {'Policy':>12s} {'Progress':>10s}")
    for name in AGENTS:
        row = latest.get(name)
        if row is None:
            print(f"{name:22s} {'—':>12s} {'—':>12s} {'—':>12s} {'—':>12s} {'waiting':>10s}")
        else:
            progress = f"{row['batch']}/{row['batches']}"
            print(f"{name:22s} {float(row['batch_loss']):12.6f} {float(row['running_loss']):12.6f} {float(row['value_loss']):12.6f} {float(row['policy_loss']):12.6f} {progress:>10s}")
    if log_lines:
        print('\nLatest training log:')
        print(''.join(log_lines[-8:]).rstrip())
    if return_code is not None:
        print(f'\nProcess exit code: {return_code}')


log_lines = []
process = subprocess.Popen(
    COMMAND, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

def collect_logs():
    assert process.stdout is not None
    for line in process.stdout:
        log_lines.append(line)

log_thread = threading.Thread(target=collect_logs, daemon=True)
log_thread.start()
try:
    while process.poll() is None:
        draw_live_loss(read_loss_rows(), read_status(), log_lines)
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    process.terminate()
    process.wait(timeout=10)
    raise
finally:
    log_thread.join(timeout=2)

draw_live_loss(read_loss_rows(), read_status(), log_lines, process.returncode)
if process.returncode != 0:
    raise RuntimeError(f'学習プロセスが失敗しました: exit={process.returncode}')